In [ ]:
import pandas as pd
from itertools import combinations
import numpy as np
from numpy.lib.stride_tricks import sliding_window_view
from scipy.stats import wasserstein_distance
import matplotlib.pyplot as plt
import pickle
np.set_printoptions(legacy='1.25')

In [ ]:
plt.rcParams["figure.figsize"] = [16, 9]
plt.rcParams["font.size"] = 20
plt.rcParams["axes.labelsize"] = 20
plt.rcParams["axes.titlesize"] = 24
plt.rcParams["xtick.labelsize"] = 16
plt.rcParams["ytick.labelsize"] = 16
plt.rcParams["font.family"] = "serif"

In [ ]:
FONT_SIZE_TITLE_PLOT = 48
FONT_SIZE_TITLE_AX = 36
FONT_SIZE_LABEL = 30
FONT_SIZE_TICKS = 24
FONT_SIZE_LEGEND = 32

In [ ]:
stock_names = ["KO", "PEP", "NVDA", "KSU"]
n_stocks = len(stock_names)

In [ ]:
seed = 42
id = "v56u7ulz"
epoch = 284
i = 0

In [ ]:
with open(f'storage/{id}/synthetic/epoch={epoch}/sample.pkl', 'rb') as f:
    d = pickle.load(f)
price_real = d['x_price']
volume_real = d['x_volume']
price_pred = d['x_hat_price']
volume_pred = d['x_hat_volume']
price_real.shape, volume_real.shape, price_pred.shape, volume_pred.shape

In [ ]:
price_real = np.concat((price_real[:, [0]], price_real), axis=1)
volume_real = np.concat((volume_real[:, [0]], volume_real), axis=1)
price_pred = np.concat((price_pred[:, [0]], price_pred), axis=1)
volume_pred = np.concat((volume_pred[:, [0]], volume_pred), axis=1)
price_real.shape, volume_real.shape, price_pred.shape, volume_pred.shape

In [ ]:
with open('storage/scaler.pkl', 'rb') as f:
    scaler = pickle.load(f)

In [ ]:
data_coscigan = np.load('storage/COSCIGAN/epoch_100.npy').transpose((0, 2, 1)).reshape(-1, 2*n_stocks)
data_coscigan = scaler.inverse_transform(data_coscigan)
data_coscigan = data_coscigan[:price_real.shape[1]].T
price_coscigan, volume_coscigan = data_coscigan[::2], np.maximum(data_coscigan[1::2], 0)
price_coscigan.shape, volume_coscigan.shape

In [ ]:
data_gtgan = np.load('storage/GT-GAN/synthetic_data_7000.npy').reshape(-1, 2*n_stocks)
data_gtgan = scaler.inverse_transform(data_gtgan)
data_gtgan = data_gtgan[:price_real.shape[1]].T
price_gtgan, volume_gtgan = data_gtgan[::2], np.maximum(data_gtgan[1::2], 0)
price_gtgan.shape, volume_gtgan.shape

In [ ]:
price_real = price_real / 10000
price_pred = price_pred / 10000
price_coscigan = price_coscigan / 10000
price_gtgan = price_gtgan / 10000

## Volume-Volatility Correlation

In [ ]:
minutes_in_a_day = 390

In [ ]:
price_real = np.reshape(price_real, shape=(n_stocks, -1, minutes_in_a_day))
price_pred = np.reshape(price_pred, shape=(n_stocks, -1, minutes_in_a_day))
price_coscigan = np.reshape(price_coscigan, shape=(n_stocks, -1, minutes_in_a_day))
price_gtgan = np.reshape(price_gtgan, shape=(n_stocks, -1, minutes_in_a_day))
price_real.shape, price_pred.shape, price_coscigan.shape, price_gtgan.shape

In [ ]:
volume_real = volume_real.reshape(n_stocks, -1, minutes_in_a_day)
volume_synthetic = volume_pred.reshape(n_stocks, -1, minutes_in_a_day)
volume_coscigan = volume_coscigan.reshape(n_stocks, -1, minutes_in_a_day)
volume_gtgan = volume_gtgan.reshape(n_stocks, -1, minutes_in_a_day)
volume_synthetic.shape, price_coscigan.shape, volume_coscigan.shape, volume_gtgan.shape

In [ ]:
window_shape = 30

In [ ]:
# Volatility computed as standard deviation of the returns on a window W over square root of W
windowed_price_real = sliding_window_view(price_real, window_shape=window_shape, axis=-1)
windowed_price_synthetic = sliding_window_view(price_pred, window_shape=window_shape, axis=-1)
windowed_price_coscigan = sliding_window_view(price_coscigan, window_shape=window_shape, axis=-1)
windowed_price_gtgan = sliding_window_view(price_gtgan, window_shape=window_shape, axis=-1)
print(windowed_price_real.shape, windowed_price_synthetic.shape, windowed_price_coscigan.shape, windowed_price_gtgan.shape)
rolled_volatility_real = (windowed_price_real.std(axis=-1) / np.sqrt(window_shape)).reshape(n_stocks, -1)
rolled_volatility_synthetic = (windowed_price_synthetic.std(axis=-1) / np.sqrt(window_shape)).reshape(n_stocks, -1)
rolled_volatility_coscigan = (windowed_price_coscigan.std(axis=-1) / np.sqrt(window_shape)).reshape(n_stocks, -1)
rolled_volatility_gtgan = (windowed_price_gtgan.std(axis=-1) / np.sqrt(window_shape)).reshape(n_stocks, -1)
print(rolled_volatility_real.shape, rolled_volatility_synthetic.shape, rolled_volatility_coscigan.shape, rolled_volatility_gtgan.shape)

In [ ]:
# Volatilty computed as square of the returns
rolled_volatility_real = np.square(price_real[:, :, window_shape - 1 :]).reshape((4, -1))
rolled_volatility_synthetic = np.square(price_pred[:, :, window_shape - 1 :]).reshape((4, -1))
rolled_volatility_coscigan = np.square(price_coscigan[:, :, window_shape - 1 :]).reshape((4, -1))
rolled_volatility_gtgan = np.square(price_gtgan[:, :, window_shape - 1 :]).reshape((4, -1))
rolled_volatility_real.shape, rolled_volatility_synthetic.shape, rolled_volatility_coscigan.shape, rolled_volatility_gtgan.shape

In [ ]:
windowed_volume_real = sliding_window_view(volume_real, window_shape=window_shape, axis=-1)
windowed_volume_synthetic = sliding_window_view(volume_synthetic, window_shape=window_shape, axis=-1)
windowed_volume_coscigan = sliding_window_view(volume_coscigan, window_shape=window_shape, axis=-1)
windowed_volume_gtgan = sliding_window_view(volume_gtgan, window_shape=window_shape, axis=-1)
print(windowed_volume_real.shape, windowed_volume_synthetic.shape, windowed_volume_coscigan.shape, windowed_volume_gtgan.shape)
rolled_mean_volume_real = (windowed_volume_real.mean(axis=-1)).reshape(n_stocks, -1)
rolled_mean_volume_synthetic = (windowed_volume_synthetic.mean(axis=-1)).reshape(n_stocks, -1)
rolled_mean_volume_coscigan = (windowed_volume_coscigan.mean(axis=-1)).reshape(n_stocks, -1)
rolled_mean_volume_gtgan = (windowed_volume_gtgan.mean(axis=-1)).reshape(n_stocks, -1)
print(rolled_mean_volume_real.shape, rolled_mean_volume_synthetic.shape, rolled_mean_volume_coscigan.shape, rolled_mean_volume_gtgan.shape)

In [ ]:
window_shape = 390 * 2
windowed_rolled_volatility_real = sliding_window_view(rolled_volatility_real, window_shape=window_shape, axis=-1)
windowed_rolled_volatility_synthetic = sliding_window_view(rolled_volatility_synthetic, window_shape=window_shape, axis=-1)
windowed_rolled_volatility_coscigan = sliding_window_view(rolled_volatility_coscigan, window_shape=window_shape, axis=-1)
windowed_rolled_volatility_gtgan = sliding_window_view(rolled_volatility_gtgan, window_shape=window_shape, axis=-1)
print(windowed_rolled_volatility_real.shape, windowed_rolled_volatility_synthetic.shape, windowed_rolled_volatility_coscigan.shape, windowed_rolled_volatility_gtgan.shape)
windowed_rolled_mean_volume_real = sliding_window_view(rolled_mean_volume_real, window_shape=window_shape, axis=-1)
windowed_rolled_mean_volume_synthetic = sliding_window_view(rolled_mean_volume_synthetic, window_shape=window_shape, axis=-1)
windowed_rolled_mean_volume_coscigan = sliding_window_view(rolled_mean_volume_coscigan, window_shape=window_shape, axis=-1)
windowed_rolled_mean_volume_gtgan = sliding_window_view(rolled_mean_volume_gtgan, window_shape=window_shape, axis=-1)
print(windowed_rolled_mean_volume_real.shape, windowed_rolled_mean_volume_synthetic.shape, windowed_rolled_mean_volume_coscigan.shape, windowed_rolled_mean_volume_gtgan.shape)

In [ ]:
d = dict()
for stock_name, real_volume, real_volatility, synthetic_volume, synthetic_volatility, cosicgan_volume, cosicgan_volatility, gtgan_volume, gtgan_volatility in zip(stock_names,
    windowed_rolled_mean_volume_real, windowed_rolled_volatility_real,
    windowed_rolled_mean_volume_synthetic, windowed_rolled_volatility_synthetic,
    windowed_rolled_mean_volume_coscigan, windowed_rolled_volatility_coscigan,
    windowed_rolled_mean_volume_gtgan, windowed_rolled_volatility_gtgan
):
    l_real = list()
    for (window_volume, window_volatility) in zip(real_volume, real_volatility):
        l_real.append(np.corrcoef(window_volume, window_volatility)[0, 1])
    l_synthetic = list()
    for (window_volume, window_volatility) in zip(synthetic_volume, synthetic_volatility):
        l_synthetic.append(np.corrcoef(window_volume, window_volatility)[0, 1])
    l_coscigan = list()
    for (window_volume, window_volatility) in zip(cosicgan_volume, cosicgan_volatility):
        l_coscigan.append(np.corrcoef(window_volume, window_volatility)[0, 1])
    l_gtgan = list()
    for (window_volume, window_volatility) in zip(gtgan_volume, gtgan_volatility):
        l_gtgan.append(np.corrcoef(window_volume, window_volatility)[0, 1])
    d[stock_name] = (np.asarray(l_real), np.asarray(l_synthetic), np.asarray(l_coscigan), np.asarray(l_gtgan))

In [ ]:
bins = np.linspace(-1, 1, 50)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 9))
axes = axes.ravel()
ns = list()
binss = list()
add_label = True
for ax, (stock_name, (corrs_real, corrs_synthetic, corrs_coscigan, corrs_gtgan)) in zip(axes, d.items()):
    ax.set_xlim((-1, 1))
    n, bins, _ = ax.hist(
        x=[corrs_gtgan, corrs_coscigan, corrs_synthetic, corrs_real],
        label=["GT-GAN", "COSCI-GAN", "CoMeTS-GAN", "Real"] if add_label else None,
        color=["C4", "C3", "C2", "C1"], bins=bins, density=True, log=True, histtype="step", linewidth=3
    )
    ns.append(n)
    binss.append(bins)
    add_label = False
    ax.set_title(f"{stock_name}", fontsize=FONT_SIZE_TITLE_AX)
    ax.set_xlabel("Correlation Coefficient", fontsize=FONT_SIZE_LABEL)
    ax.set_ylabel("Density", fontsize=FONT_SIZE_LABEL)
    ax.xaxis.set_tick_params(labelsize=FONT_SIZE_TICKS)
    ax.yaxis.set_tick_params(labelsize=FONT_SIZE_TICKS)
    ax.set_facecolor('#eaeaf2')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['bottom'].set_visible(False)
    ax.spines['left'].set_visible(False)
    ax.tick_params(bottom=False, left=False, which='both')
    ax.grid(True, linestyle='solid', c='w')

fig.legend(loc="upper center", ncol=4, fontsize=FONT_SIZE_LEGEND-2, frameon=False)
fig.tight_layout(rect=[0, 0, 1, 0.9])
# plt.savefig("plots/volume_volatility_correlation_sota.pdf")
plt.show()
plt.close(fig)

## Correlations

In [ ]:
price_real = price_real.transpose(1, 0, 2)
price_pred = price_pred.transpose(1, 0, 2)
price_coscigan = price_coscigan.transpose(1, 0, 2)
price_gtgan = price_gtgan.transpose(1, 0, 2)

In [ ]:
ind = np.triu_indices(n_stocks, k=1)
corrs_real = np.asarray([np.corrcoef(p)[ind] for p in price_real]).T
corrs_pred = np.asarray([np.corrcoef(p)[ind] for p in price_pred]).T
corrs_coscigan = np.asarray([np.corrcoef(p)[ind] for p in price_coscigan]).T
corrs_gtgan = np.asarray([np.corrcoef(p)[ind] for p in price_gtgan]).T
corrs_real.shape, corrs_pred.shape, corrs_coscigan.shape, corrs_gtgan.shape

In [ ]:
stock_pairs = list(combinations(stock_names, 2))
fig, axes = plt.subplots(3, 6, figsize=(16, 9))
add_label = True
wass = dict()
wass['CoMeTS-GAN'] = dict()
wass['COSCI-GAN'] = dict()
wass['GT-GAN'] = dict()
for (ax1, ax2, ax3), corr_real, corr_pred, corr_coscigan, corr_gtgan, (s1, s2) in zip(axes.T, corrs_real, corrs_pred, corrs_coscigan, corrs_gtgan, stock_pairs):
    title = f'{s1} - {s2}'
    ax1.set_title(title)

    n, _, _ = ax1.hist(
        [corr_pred, corr_real], 
        label=["CoMeTS-GAN", "Real"] if add_label else None,
        color=["C2", "C1"],
        density=True, histtype="step", linewidth=2, 
    )
    wass['CoMeTS-GAN'][title] = wasserstein_distance(*n)
    n, _, _ = ax2.hist(
        [corr_coscigan, corr_real], 
        label=["COSCI-GAN", "Real"] if add_label else None,
        color=["C3", "C1"],
        density=True, histtype="step", linewidth=2, 
    )
    wass['COSCI-GAN'][title] = wasserstein_distance(*n)
    n, _, _ = ax3.hist(
        [corr_gtgan, corr_real], 
        label=["GT-GAN", "Real"] if add_label else None,
        color=["C4", "C1"],
        density=True, histtype="step", linewidth=2, 
    )
    wass['GT-GAN'][title] = wasserstein_distance(*n)
    ax1.hist(
        [corr_pred, corr_real], 
        color=["C2", "C1"],
        density=True, histtype="stepfilled", alpha=.3, 
    )
    ax2.hist(
        [corr_coscigan, corr_real], 
        color=["C3", "C1"],
        density=True, histtype="stepfilled", alpha=.3, 
    )
    ax3.hist(
        [corr_gtgan, corr_real], 
        color=["C4", "C1"],
        density=True, histtype="stepfilled", alpha=.3, 
    )
    add_label = False

axes[0][0].set_ylabel('Density')
axes[1][0].set_ylabel('Density')
axes[2][0].set_ylabel('Density')

for ax in axes.ravel():
    ax.set_xlabel(r'$\rho$')
    ax.set_xlim((-1.1, 1.1))
    ax.set_ylim((0, 6))
    ax.set_facecolor('#eaeaf2')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['bottom'].set_visible(False)
    ax.spines['left'].set_visible(False)
    ax.tick_params(bottom=False, left=False, which='both')
    ax.grid(True, linestyle='solid', c='w')

# remove duplicate labels from legend
handles0, labels0 = axes[0][0].get_legend_handles_labels()
handles1, labels1 = axes[1][0].get_legend_handles_labels()
handles2, labels2 = axes[2][0].get_legend_handles_labels()
by_label = dict(zip(labels0 + labels1 + labels2, handles0 + handles1 + handles2))
fig.legend(
    handles=by_label.values(), labels=by_label.keys(), 
    loc="upper center", ncol=4, fontsize=FONT_SIZE_LEGEND, frameon=False, bbox_to_anchor=(0.5, 1.1))
fig.tight_layout()
# plt.savefig(f'plots/correlations_sota.pdf', bbox_inches='tight')
plt.show()
plt.close(fig)

In [ ]:
df = pd.DataFrame(wass)
df

In [ ]:
df.to_latex(
    'correlations_sota.tex', float_format="%.2f", 
    caption="Wasserstein distances between the correlation distributions.",
    label="tab:correlations_sota_wass",
)